# Nemotron Reasoning — Baseline SFT Notebook w/ Unsloth

Unsloth is an optimization library that makes finetuning faster and uses lower VRAM. Using Unsloth, we can actually train on the whole dataset within a reasonable time (~1hr/epoch).

Fun fact: Unsloth has an actual guide for Nemotron 3 models - https://unsloth.ai/docs/models/nemotron-3

For training with this notebook, you will need a bunch of libraries other than Unsloth, the most important of which is mamba-ssm

## 1. Setup

In [ ]:
!pip install -q --no-index --find-links /kaggle/input/datasets/mayukh18/nemotron-packages/packages unsloth trl peft transformers datasets accelerate bitsandbytes 
!pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
!pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

In [ ]:
import os
import re
import math
from pathlib import Path
import zipfile
import pandas as pd
from datasets import Dataset
import kagglehub
import torch
from torch import nn
import bitsandbytes as bnb
from unsloth import FastLanguageModel
from sklearn.model_selection import train_test_split

DATA_DIR = Path("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH  = DATA_DIR / "test.csv"
ADAPTER_DIR = "nemotron-lora-adapter"
SUBMISSION_ZIP = "submission.zip"

FRESH_START = True

if FRESH_START:
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
else:
    MODEL_PATH = "/kaggle/input/datasets/mayukh18/nemotron-lora-submission/submission/nemotron-lora-adapter"

# LoRA config
LORA_RANK    = 32      # competition allows up to 32
LORA_ALPHA   = 32
LORA_DROPOUT = 0.0     # 0 enables full Unsloth fast-kernel patching on LoRA matrices

# Training config
MAX_SEQ_LEN = 1024
NUM_EPOCHS  = 1
BATCH_SIZE  = 6
GRAD_ACCUM  = 2
LR          = 1e-4

MAX_NEW_TOKENS = 512
TEMPERATURE    = 0.0

print("Config ready.")

## 2. Load & Inspect Data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

train_df, val_df = train_test_split(train_df, test_size=100, random_state=42)

print(f"Train: {len(train_df):,} rows — columns: {list(train_df.columns)}")
print(f"Test:  {len(val_df):,} rows  — columns: {list(val_df.columns)}")
print(f"Test:  {len(test_df):,} rows  — columns: {list(test_df.columns)}")
train_df.head()

In [ ]:
# Quick look at prompt length distribution
train_df["prompt_len"] = train_df["prompt"].str.len()
print(train_df["prompt_len"].describe())

# Sample one puzzle
sample = train_df.sample(1).iloc[0]
print("\n--- Sample Prompt ---")
print(sample["prompt"])
print("\n--- Answer ---")
print(sample["answer"])

## 3. Format Data

We mirror the exact prompt format used in the competition's evaluation metric so there is **no train/inference mismatch**.

The metric notebook appends this instruction to every user message:
```
\nPlease put your final answer inside `\boxed{}`. For example: `\boxed{your answer}`
```

For training, the assistant response is simply `\boxed{answer}`.  
A chain-of-thought prefix can be added in a later iteration.

In [ ]:
BOXED_INSTRUCTION = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)

def create_answer_str(answer):
    safe = str(answer).replace("}", "\\}")
    return f"<think>\nLet me work through this step by step.\n</think>\n\\boxed{{{safe}}}"

def format_train_row(prompt: str, answer: str) -> dict:
    """Return a messages list for SFT chat-template formatting."""
    return {
        "messages": [
            {"role": "user",      "content": prompt + BOXED_INSTRUCTION},
            {"role": "assistant", "content": create_answer_str(answer)},
        ]
    }

def format_test_row(prompt: str) -> dict:
    """Return a messages list for inference (no answer)."""
    return {
        "messages": [
            {"role": "user", "content": prompt + BOXED_INSTRUCTION},
        ]
    }

# Build HuggingFace Dataset for training
train_records = [
    format_train_row(row["prompt"], str(row["answer"]))
    for _, row in train_df.iterrows()
]
train_dataset = Dataset.from_list(train_records)

# Build HuggingFace Dataset for validation
val_records = [
    format_train_row(row["prompt"], str(row["answer"]))
    for _, row in val_df.iterrows()
]
val_dataset = Dataset.from_list(val_records)

print(f"Formatted {len(train_dataset):,} train examples, {len(val_dataset):,} val examples.")
print("Sample messages:\n", train_dataset[0]["messages"])

## 4. Load Base Model with LoRA (4-bit via Unsloth)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = MAX_SEQ_LEN, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    trust_remote_code = True,
    unsloth_force_compile = False,
    attn_implementation = "eager",
    torch_dtype=torch.bfloat16,
    dtype=None,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

In [ ]:
# del model, tokenizer

# import gc
# gc.collect()
# torch.cuda.empty_cache()

In [ ]:
target_modules = [
    'out_proj', 'v_proj', 'q_proj', 'down_proj', 'embed_tokens',
    'k_proj', 'in_proj', 'up_proj', 'o_proj', 'lm_head', 'gate_proj'
]

In [ ]:
if FRESH_START:
    # Apply LoRA adapter
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=target_modules,
        bias="none",
        use_gradient_checkpointing="unsloth",  # saves VRAM
        random_state=42,
    )

model.print_trainable_parameters()

## 5. Train with SFTTrainer

In [ ]:
from transformers import TrainerCallback
from unsloth import FastLanguageModel

class SampleGenCallback(TrainerCallback):
    def __init__(self, val_df, tokenizer, n=3):
        self.samples = val_df.head(n).reset_index(drop=True)
        self.tokenizer = tokenizer

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        FastLanguageModel.for_inference(model)
        print(f"\n{'='*60}")
        print(f"Step {state.global_step} — sample generations")
        print('='*60)
        for _, row in self.samples.iterrows():
            messages = [{"role": "user", "content": row["prompt"] + BOXED_INSTRUCTION}]
            text = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=True,
            )
            input_ids = self.tokenizer(text, return_tensors="pt").input_ids.to(model.device)
            output_ids = model.generate(
                input_ids=input_ids,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
            )
            generated = self.tokenizer.decode(
                output_ids[0][input_ids.shape[1]:], skip_special_tokens=True
            )
            print(f"\nPrompt : {row['prompt'][:80]}...")
            print(f"Gold   : {row['answer']}")
            print(f"Output : {generated[:200]}")
        FastLanguageModel.for_training(model)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import train_on_responses_only

train_dataset = train_dataset.map(lambda ex: {
    "text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False,
        enable_thinking=True,
    )
})

val_dataset = val_dataset.map(lambda ex: {
    "text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False,
        enable_thinking=True,
    )
})

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=TrainingArguments(
        output_dir="./nemotron-lora-checkpoints",
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        bf16=True,               # use fp16=True if bf16 unavailable
        logging_steps=50,
        save_strategy="epoch",
        save_total_limit=1,
        optim="adamw_8bit",      # 8-bit Adam from bitsandbytes
        seed=42,
        report_to="none",        # disable wandb/tensorboard for baseline
        dataloader_num_workers=4,
        eval_strategy="steps",
        eval_steps=50,
    ),
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    dataset_kwargs={"skip_prepare_dataset": False},
    callbacks=[SampleGenCallback(val_df, tokenizer, n=3)],
)

# Mask prompt tokens — compute loss only on assistant response
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

# Sanity check: confirm only assistant response tokens have active loss
sample = next(iter(trainer.get_train_dataloader()))
active = (sample["labels"][0] != -100).nonzero().flatten()
print("Active (loss-bearing) tokens:")
print(tokenizer.decode(sample["input_ids"][0][active]))
print()

In [ ]:
# example_text = tokenizer.apply_chat_template(
#     [
#         {"role": "user",      "content": "test prompt"},
#         {"role": "assistant", "content": "test response"},
#     ],
#     tokenize=False,
#     add_generation_prompt=False,
#     enable_thinking=True,
# )
# print(repr(example_text))

In [ ]:
print("Starting training...")
trainer.train()
print("Training complete.")

## 6. Save LoRA Adapter

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Verify adapter_config.json is present (required by competition)
assert os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")), \
    "adapter_config.json missing!"

print(f"Adapter saved to ./{ADAPTER_DIR}/")
print("Files:", os.listdir(ADAPTER_DIR))

In [ ]:
# def inspect_batch(batch, label, n=2):
#     print(f"\n=== {label} ===")
#     for i in range(min(n, len(batch["input_ids"]))):
#         ids    = batch["input_ids"][i]
#         labels = batch["labels"][i]
#         active = (labels != -100).nonzero().flatten()
#         masked = (labels == -100).nonzero().flatten()
#         print(f"Sample {i}: {len(masked)} masked tokens, {len(active)} active tokens")
#         print(f"  Active text: {repr(tokenizer.decode(ids[active]))}")

# train_batch = next(iter(trainer.get_train_dataloader()))
# eval_batch  = next(iter(trainer.get_eval_dataloader()))

# inspect_batch(train_batch, "TRAIN")
# inspect_batch(eval_batch,  "EVAL")

## 7. Local Validation

Run inference on a small held-out slice of the training set to get a quick local accuracy estimate before submitting.

In [ ]:
def extract_final_answer(text: str | None) -> str:
    r"""Extract the final answer from model output, prioritising \boxed{}."""
    if text is None:
        return "NOT_FOUND"

    # Prefer \boxed{...}
    matches = re.findall(r"\\boxed\{([^}]*)(?:\}|$)", text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        return non_empty[-1] if non_empty else matches[-1].strip()

    # Common fallback patterns
    patterns = [
        r"The final answer is:\s*([^\n]+)",
        r"Final answer is:\s*([^\n]+)",
        r"Final answer\s*[:：]\s*([^\n]+)",
    ]
    for pattern in patterns:
        m = re.findall(pattern, text, re.IGNORECASE)
        if m:
            return m[-1].strip()

    # Last numeric value
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    if nums:
        return nums[-1]

    # Last non-empty line
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return lines[-1] if lines else "NOT_FOUND"


def verify(stored_answer: str, predicted: str) -> bool:
    """Return True if predicted matches stored_answer (numeric or string)."""
    stored_answer = stored_answer.strip()
    predicted = predicted.strip()
    try:
        return math.isclose(float(stored_answer), float(predicted),
                            rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == stored_answer.lower()


print("Helper functions ready.")

In [ ]:
# # Switch model to inference mode (Unsloth optimisation)
# FastLanguageModel.for_inference(model)

# # Sample 100 examples from train for local eval
# eval_df = train_df.sample(20, random_state=42).reset_index(drop=True)

# correct = 0
# for _, row in eval_df.iterrows():
#     user_content = row["prompt"] + BOXED_INSTRUCTION
#     # Apply the tokenizer's chat template (system prompt optional for baseline)
#     messages = [{"role": "user", "content": user_content}]

#     text = tokenizer.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)

#     output_ids = model.generate(
#         input_ids=input_ids,
#         max_new_tokens=MAX_NEW_TOKENS,
#         temperature=TEMPERATURE if TEMPERATURE > 0 else None,
#         do_sample=TEMPERATURE > 0,
#         pad_token_id=tokenizer.eos_token_id,
#     )
#     # Decode only newly generated tokens
#     generated = tokenizer.decode(
#         output_ids[0][input_ids.shape[1]:],
#         skip_special_tokens=True
#     )

#     pred = extract_final_answer(generated)
#     if verify(str(row["answer"]), pred):
#         correct += 1

# local_acc = correct / len(eval_df)
# print(f"Local accuracy on {len(eval_df)} train samples: {local_acc:.2%}")

## 8. Submission

The competition expects a zip archive containing the LoRA adapter directory (with `adapter_config.json` at the root of the archive or a sub-directory).

In [ ]:
!rm -rf /kaggle/working/nemotron-lora-checkpoints

In [ ]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(ADAPTER_DIR):
        zf.write(
            os.path.join(ADAPTER_DIR, fname),
            arcname=os.path.join(ADAPTER_DIR, fname),
        )

# Verify
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zf:
    names = zf.namelist()

has_config = any("adapter_config.json" in n for n in names)
print(f"Files in {SUBMISSION_ZIP}: {names}")
print(f"adapter_config.json present: {has_config}")